# Airline Flight Delay Analytics & Prediction

**Big Data Analytics (BDA) project — B.E. CSE, SFIT Mumbai**

This notebook is a Colab-runnable, narrated consolidation of the project's full
PySpark pipeline, run on a **combined 2015+2016 dataset** (11.25M cleaned
flights, 1.3GB raw) to meet a GB-scale data requirement. It ingests and joins
two years of Kaggle flight-delay CSVs using an explicit schema, cleans the
data and builds a FAA-standard `is_delayed` label, aggregates delay rates
across airline / airport / hour / day-of-week / month, visualizes the
findings, and trains a class-imbalance-aware Random Forest classifier to
predict delays from information known before departure. Every section mirrors
the equivalent verified script under `src/` (`ingest_v2.py` → `clean_v2.py` →
`aggregate_v2.py` → `eda_v2.py` → `model_v2.py`) — this notebook adapts paths
for Colab's `/content/` filesystem and adds narration, but does not diverge
from the pipeline logic already validated locally.

An **Appendix** at the end preserves the project's original single-year
(2015-only, 5.8M rows) pipeline run as a baseline — the project started small,
validated the approach, then scaled up to the combined dataset without the
findings or model performance meaningfully changing. That stability is itself
one of the findings (see the Conclusion).

## 1. Setup

This notebook is designed to run top-to-bottom on **Google Colab**.

**Colab vs. local setup:** Colab's runtime already ships a Java Runtime
Environment, which PySpark needs under the hood to launch its JVM gateway —
so unlike the local development environment for this project (which required
installing `openjdk@17` via Homebrew on macOS before PySpark would run at
all), no Java installation step is needed here. We only need to install
PySpark itself.

In [ ]:
!pip install -q pyspark

## 2. Download the datasets

This notebook needs your Kaggle API credentials. Two ways to provide them:

1. **Upload `kaggle.json` directly** — run the upload cell below and use the
   file picker. Generate `kaggle.json` at https://www.kaggle.com/settings →
   API → "Create New Token".
2. **Colab Secrets manager** — store your Kaggle username/key as Colab
   secrets (`KAGGLE_USERNAME`, `KAGGLE_KEY`) via the 🔑 icon in the left
   sidebar instead, then skip the upload cell and use the commented
   `userdata` alternative in the cell after it.

We need two Kaggle datasets:
- **`usdot/flight-delays`** — small (~700MB), downloaded whole. Provides
  `airlines.csv`/`airports.csv` (the IATA-code lookup tables used by every
  ingest step in this notebook, primary and appendix) plus the original
  `flights.csv` used by the Appendix baseline.
- **`yuanyuwendymu/airline-delay-and-cancellation-data-2009-2018`** — this is
  the combined-year source, but it's ~7GB uncompressed across all 10 years
  (2009-2018).

### Why `-f` instead of the full multi-year zip
Downloading and extracting that full 7GB archive **crashed a local disk in an
earlier session** — the extraction ran out of space mid-write and corrupted
the file being written (`2018.csv` was left truncated mid-row). We only need
2 of the 10 years to comfortably clear a 1GB+ scale requirement (2015+2016
alone is 1.3GB, 11.4M rows), so instead we use Kaggle's `-f` flag to fetch
`2015.csv` and `2016.csv` individually — far safer and faster than
downloading+extracting the full archive just to use two files from it.

In [ ]:
from google.colab import files

# Upload the kaggle.json file you downloaded from https://www.kaggle.com/settings
# (skip this cell if you're using Colab Secrets instead — see markdown above)
uploaded = files.upload()

In [ ]:
import os

os.makedirs("/root/.kaggle", exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

# Alternative if using Colab Secrets instead of the file-upload cell above:
# from google.colab import userdata
# os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
# os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

!pip install -q kaggle

# Small lookup-table + Appendix-baseline dataset (whole dataset, ~700MB, safe to download in full)
!mkdir -p /content/data/raw
!kaggle datasets download -d usdot/flight-delays -p /content/data/raw --unzip
!ls -la /content/data/raw

In [ ]:
# Combined-year dataset: fetch ONLY 2015.csv and 2016.csv via -f, not the full
# 7GB zip (see markdown above for why).
!mkdir -p /content/data/raw_v2
!kaggle datasets download -d yuanyuwendymu/airline-delay-and-cancellation-data-2009-2018 -f 2015.csv -p /content/data/raw_v2 --unzip
!kaggle datasets download -d yuanyuwendymu/airline-delay-and-cancellation-data-2009-2018 -f 2016.csv -p /content/data/raw_v2 --unzip

# Kaggle's --unzip sometimes leaves single-file (-f) downloads as "<name>.csv.zip"
# instead of extracting them (observed locally) -- unzip defensively if so.
import glob
import zipfile

for zpath in glob.glob("/content/data/raw_v2/*.csv.zip"):
    with zipfile.ZipFile(zpath) as zf:
        zf.extractall("/content/data/raw_v2")
    os.remove(zpath)

!ls -la /content/data/raw_v2

## 3. Ingest — combined 2015+2016 dataset (schema mismatch discovered & handled)

We read `2015.csv` and `2016.csv` with an explicit `StructType` (never
`inferSchema` on files this size) and union them into one dataframe, then
left-join airline full names and origin airport city/state exactly as
before.

### Data quality finding: this dataset's schema is NOT the same as the original
Despite covering the same real-world 2015 flights as `usdot/flight-delays`
(verified: this source's 2015-only row count matches exactly, 5,819,079),
the `yuanyuwendymu/...` repackaging uses **completely different column
names**:

| Original (`usdot/flight-delays`) | Combined source (`yuanyuwendymu/...`) |
|---|---|
| `YEAR`/`MONTH`/`DAY`/`DAY_OF_WEEK` | `FL_DATE` (single date string — **no day-of-week column at all**) |
| `AIRLINE` | `OP_CARRIER` |
| `ORIGIN_AIRPORT` / `DESTINATION_AIRPORT` | `ORIGIN` / `DEST` |
| `SCHEDULED_DEPARTURE` / `SCHEDULED_ARRIVAL` | `CRS_DEP_TIME` / `CRS_ARR_TIME` |
| `ARRIVAL_DELAY` | `ARR_DELAY` |
| `CANCELLATION_REASON` | `CANCELLATION_CODE` |
| `AIR_SYSTEM_DELAY` / `AIRLINE_DELAY` | `NAS_DELAY` / `CARRIER_DELAY` |
| — | `Unnamed: 27` (junk trailing column from a trailing comma in the source CSV) |

Applying the original `FLIGHTS_SCHEMA` positionally to these files would
silently corrupt every column — e.g. the `FL_DATE` string forced into an
`IntegerType` `YEAR` slot. Instead we define a schema matching the *actual*
columns, then rename/derive everything to match the original pipeline's
naming convention, including **deriving `DAY_OF_WEEK` from `FL_DATE`**. That
derivation hit a second surprise: Spark 3.0 removed the legacy `'u'`
ISO-weekday `date_format` pattern, so we compute it arithmetically from
`dayofweek()` instead (`dayofweek()` returns Sunday=1..Saturday=7; we convert
to ISO Monday=1..Sunday=7). Once renamed, every downstream cell in this
notebook is identical regardless of which raw source the data came from.

We also check whether the combined data contains any `AIRLINE` codes not
covered by our 14-row `airlines.csv` lookup (2016 could in principle include
a carrier absent from a 2015-only lookup table) and print any unmatched codes
rather than silently leaving them with a null `AIRLINE_NAME`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    DoubleType,
)

RAW_DIR = "/content/data/raw"          # airlines.csv / airports.csv lookup tables
RAW_V2_DIR = "/content/data/raw_v2"    # 2015.csv / 2016.csv (combined source)
PROCESSED_DIR = "/content/data/processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

YEAR_FILES = ["2015.csv", "2016.csv"]

# Matches raw_v2's ACTUAL header order/types -- NOT the same as the original
# usdot FLIGHTS_SCHEMA (see markdown above).
RAW_V2_SCHEMA = StructType(
    [
        StructField("FL_DATE", StringType(), True),
        StructField("OP_CARRIER", StringType(), True),
        StructField("OP_CARRIER_FL_NUM", IntegerType(), True),
        StructField("ORIGIN", StringType(), True),
        StructField("DEST", StringType(), True),
        StructField("CRS_DEP_TIME", IntegerType(), True),
        StructField("DEP_TIME", DoubleType(), True),
        StructField("DEP_DELAY", DoubleType(), True),
        StructField("TAXI_OUT", DoubleType(), True),
        StructField("WHEELS_OFF", DoubleType(), True),
        StructField("WHEELS_ON", DoubleType(), True),
        StructField("TAXI_IN", DoubleType(), True),
        StructField("CRS_ARR_TIME", IntegerType(), True),
        StructField("ARR_TIME", DoubleType(), True),
        StructField("ARR_DELAY", DoubleType(), True),
        StructField("CANCELLED", DoubleType(), True),
        StructField("CANCELLATION_CODE", StringType(), True),
        StructField("DIVERTED", DoubleType(), True),
        StructField("CRS_ELAPSED_TIME", DoubleType(), True),
        StructField("ACTUAL_ELAPSED_TIME", DoubleType(), True),
        StructField("AIR_TIME", DoubleType(), True),
        StructField("DISTANCE", DoubleType(), True),
        StructField("CARRIER_DELAY", DoubleType(), True),
        StructField("WEATHER_DELAY", DoubleType(), True),
        StructField("NAS_DELAY", DoubleType(), True),
        StructField("SECURITY_DELAY", DoubleType(), True),
        StructField("LATE_AIRCRAFT_DELAY", DoubleType(), True),
        StructField("UNNAMED_27", StringType(), True),  # trailing junk column, always blank
    ]
)

AIRLINES_SCHEMA = StructType(
    [
        StructField("IATA_CODE", StringType(), True),
        StructField("AIRLINE", StringType(), True),
    ]
)

AIRPORTS_SCHEMA = StructType(
    [
        StructField("IATA_CODE", StringType(), True),
        StructField("AIRPORT", StringType(), True),
        StructField("CITY", StringType(), True),
        StructField("STATE", StringType(), True),
        StructField("COUNTRY", StringType(), True),
        StructField("LATITUDE", DoubleType(), True),
        StructField("LONGITUDE", DoubleType(), True),
    ]
)


def rename_to_pipeline_schema(df):
    """Rename/derive raw_v2 columns to match the original pipeline's naming
    convention, so every downstream cell stays identical regardless of source."""
    return (
        df.withColumn("YEAR", F.year("FL_DATE"))
        .withColumn("MONTH", F.month("FL_DATE"))
        .withColumn("DAY", F.dayofmonth("FL_DATE"))
        # ISO day-of-week (Monday=1..Sunday=7). dayofweek() returns Sunday=1..
        # Saturday=7; the 'u' date_format pattern would give ISO directly but
        # was removed in Spark 3.0, so convert arithmetically instead.
        .withColumn("DAY_OF_WEEK", ((F.dayofweek("FL_DATE") + 5) % 7) + 1)
        .withColumnRenamed("OP_CARRIER", "AIRLINE")
        .withColumnRenamed("OP_CARRIER_FL_NUM", "FLIGHT_NUMBER")
        .withColumnRenamed("ORIGIN", "ORIGIN_AIRPORT")
        .withColumnRenamed("DEST", "DESTINATION_AIRPORT")
        .withColumnRenamed("CRS_DEP_TIME", "SCHEDULED_DEPARTURE")
        .withColumnRenamed("DEP_TIME", "DEPARTURE_TIME")
        .withColumnRenamed("DEP_DELAY", "DEPARTURE_DELAY")
        .withColumnRenamed("CRS_ELAPSED_TIME", "SCHEDULED_TIME")
        .withColumnRenamed("ACTUAL_ELAPSED_TIME", "ELAPSED_TIME")
        .withColumn("DISTANCE", F.col("DISTANCE").cast("int"))
        .withColumnRenamed("CRS_ARR_TIME", "SCHEDULED_ARRIVAL")
        .withColumnRenamed("ARR_TIME", "ARRIVAL_TIME")
        .withColumnRenamed("ARR_DELAY", "ARRIVAL_DELAY")
        .withColumn("DIVERTED", F.col("DIVERTED").cast("int"))
        .withColumn("CANCELLED", F.col("CANCELLED").cast("int"))
        .withColumnRenamed("CANCELLATION_CODE", "CANCELLATION_REASON")
        .withColumnRenamed("CARRIER_DELAY", "AIRLINE_DELAY")
        .withColumnRenamed("NAS_DELAY", "AIR_SYSTEM_DELAY")
        .drop("FL_DATE", "UNNAMED_27")
    )


spark = (
    SparkSession.builder.appName("AirlineDelayAnalytics-Ingest-V2")
    .master("local[*]")
    # 4g driver memory: the aggregation and modeling steps later in this
    # notebook shuffle/persist enough data (11M+ rows, twice the single-year
    # baseline) that the default 1g driver heap runs out. This one
    # SparkSession is reused for the whole notebook, appendix included.
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

year_dfs = [
    spark.read.csv(f"{RAW_V2_DIR}/{f}", header=True, schema=RAW_V2_SCHEMA)
    for f in YEAR_FILES
]
flights_raw = year_dfs[0]
for extra_df in year_dfs[1:]:
    flights_raw = flights_raw.unionByName(extra_df)

flights = rename_to_pipeline_schema(flights_raw)

airlines = spark.read.csv(f"{RAW_DIR}/airlines.csv", header=True, schema=AIRLINES_SCHEMA)
airports = spark.read.csv(f"{RAW_DIR}/airports.csv", header=True, schema=AIRPORTS_SCHEMA)

# Check for AIRLINE codes not covered by the 2015-only airlines.csv lookup --
# print them rather than silently dropping/nulling them.
known_codes = [row["IATA_CODE"] for row in airlines.select("IATA_CODE").collect()]
unmatched = (
    flights.filter(~F.col("AIRLINE").isin(known_codes))
    .groupBy("AIRLINE")
    .count()
    .orderBy(F.desc("count"))
    .collect()
)
if unmatched:
    print(f"\nWARNING: {len(unmatched)} AIRLINE code(s) are NOT in airlines.csv's {len(known_codes)} entries:")
    for row in unmatched:
        print(f"  {row['AIRLINE']}: {row['count']} rows (will have null AIRLINE_NAME after the left join)")
else:
    print(f"\nAll AIRLINE codes in the combined data are covered by airlines.csv's {len(known_codes)} entries.")

airlines_renamed = airlines.withColumnRenamed("AIRLINE", "AIRLINE_NAME")
joined = flights.join(
    airlines_renamed, flights["AIRLINE"] == airlines_renamed["IATA_CODE"], "left"
).drop(airlines_renamed["IATA_CODE"])

origin_airports = (
    airports.withColumnRenamed("CITY", "ORIGIN_CITY")
    .withColumnRenamed("STATE", "ORIGIN_STATE")
    .select("IATA_CODE", "ORIGIN_CITY", "ORIGIN_STATE")
)
joined = joined.join(
    origin_airports, joined["ORIGIN_AIRPORT"] == origin_airports["IATA_CODE"], "left"
).drop(origin_airports["IATA_CODE"])

row_count = joined.count()
print(f"\nJoined row count (2015+2016 combined): {row_count}\n")
print("Joined schema:")
joined.printSchema()

joined.write.mode("overwrite").parquet(f"{PROCESSED_DIR}/flights_joined_v2.parquet")

print("\nSample rows:")
joined.select(
    "YEAR", "MONTH", "DAY", "DAY_OF_WEEK", "AIRLINE", "AIRLINE_NAME",
    "ORIGIN_AIRPORT", "ORIGIN_CITY", "ORIGIN_STATE",
    "DESTINATION_AIRPORT", "ARRIVAL_DELAY",
).show(5, truncate=False)

## 4. Clean — filter invalid flights, build the `is_delayed` label

Identical logic to the ingest→clean handoff used throughout this project:
drop `CANCELLED`/`DIVERTED` flights (no meaningful `ARRIVAL_DELAY` to
predict), label `is_delayed = 1` if `ARRIVAL_DELAY >= 15` minutes (the FAA
standard), fill the five `*_DELAY` cause columns' nulls with `0` (null means
"no delay of that type," not missing data), and derive
`SCHEDULED_DEPARTURE_HOUR`.

In [ ]:
DELAY_CAUSE_COLUMNS = [
    "AIR_SYSTEM_DELAY",
    "SECURITY_DELAY",
    "AIRLINE_DELAY",
    "LATE_AIRCRAFT_DELAY",
    "WEATHER_DELAY",
]

df = spark.read.parquet(f"{PROCESSED_DIR}/flights_joined_v2.parquet")

total_rows = df.count()
print(f"\nStarting row count: {total_rows}")

cancelled_count = df.filter(F.col("CANCELLED") == 1).count()
diverted_count = df.filter((F.col("CANCELLED") != 1) & (F.col("DIVERTED") == 1)).count()
print(f"Rows dropped for CANCELLED == 1: {cancelled_count}")
print(f"Rows dropped for DIVERTED == 1 (excluding already-cancelled): {diverted_count}")

df = df.filter((F.col("CANCELLED") != 1) & (F.col("DIVERTED") != 1))
after_filter_rows = df.count()
print(f"Row count after dropping cancelled/diverted: {after_filter_rows}")

df = df.fillna(0, subset=DELAY_CAUSE_COLUMNS)

df = df.withColumn("is_delayed", F.when(F.col("ARRIVAL_DELAY") >= 15, 1).otherwise(0))

delayed_count = df.filter(F.col("is_delayed") == 1).count()
on_time_count = after_filter_rows - delayed_count
pct_delayed = 100.0 * delayed_count / after_filter_rows
pct_on_time = 100.0 * on_time_count / after_filter_rows
print(
    f"\nClass balance — delayed: {delayed_count} ({pct_delayed:.2f}%), "
    f"on-time: {on_time_count} ({pct_on_time:.2f}%)"
)

df = df.withColumn(
    "SCHEDULED_DEPARTURE_HOUR", (F.col("SCHEDULED_DEPARTURE") / 100).cast("int")
)

final_row_count = df.count()
print(f"\nFinal row count: {final_row_count}")

df.write.mode("overwrite").parquet(f"{PROCESSED_DIR}/flights_clean_v2.parquet")

print("\nSample rows:")
df.select(
    "YEAR", "MONTH", "DAY", "AIRLINE", "SCHEDULED_DEPARTURE",
    "SCHEDULED_DEPARTURE_HOUR", "ARRIVAL_DELAY", "is_delayed",
    "AIR_SYSTEM_DELAY", "SECURITY_DELAY", "AIRLINE_DELAY",
    "LATE_AIRCRAFT_DELAY", "WEATHER_DELAY",
).show(5, truncate=False)

## 5. Aggregate — delay rates by airline, airport, hour, day-of-week, month

Same aggregation logic and the same defensive IATA-code filter for the
by-airport view as the original pipeline.

> **Data quality comparison point:** the original `usdot/flight-delays`
> source has all of October 2015 (~8.4% of rows, 482,878 flights) using
> numeric DOT airport IDs instead of 3-letter IATA codes — a quirk we found
> and worked around by filtering `ORIGIN_AIRPORT` to `^[A-Z]{3}$` for the
> airport-keyed aggregation only. Running that same defensive filter against
> **this combined source finds 0 rows to exclude** — this Kaggle
> repackaging (`yuanyuwendymu/...`) does not carry the same data quality
> issue, even though it covers the same underlying flights. Worth noting for
> the report: identical real-world data, different quirks depending on how a
> given Kaggle uploader repackaged it — a good example of why raw data
> always needs verifying rather than trusting column names/formats to be
> consistent across sources.

In [ ]:
import json

from pyspark import StorageLevel

METRICS_DIR = "/content/outputs/metrics"
os.makedirs(METRICS_DIR, exist_ok=True)

DELAY_CAUSE_COLS_AGG = [
    "AIRLINE_DELAY", "WEATHER_DELAY", "AIR_SYSTEM_DELAY",
    "SECURITY_DELAY", "LATE_AIRCRAFT_DELAY",
]


def delay_rate_agg(frame, group_cols):
    return (
        frame.groupBy(*group_cols)
        .agg(F.count("*").alias("total_flights"), F.sum("is_delayed").alias("total_delayed"))
        .withColumn("delay_pct", F.round(100.0 * F.col("total_delayed") / F.col("total_flights"), 2))
    )


def save(frame, name):
    frame.coalesce(1).write.mode("overwrite").parquet(f"{PROCESSED_DIR}/{name}.parquet")
    frame.coalesce(1).write.mode("overwrite").option("header", True).csv(f"{PROCESSED_DIR}/{name}.csv")


df.persist(StorageLevel.DISK_ONLY)

# a. By AIRLINE_NAME
by_airline = delay_rate_agg(df, ["AIRLINE_NAME"]).orderBy(F.desc("delay_pct"))
save(by_airline, "agg_by_airline_v2")

# b. By ORIGIN_AIRPORT, IATA-only + busy airports (see data quality note above)
IATA_CODE_PATTERN = "^[A-Z]{3}$"
airport_rows_before = df.select("ORIGIN_AIRPORT").count()
valid_airport_df = df.filter(F.col("ORIGIN_AIRPORT").rlike(IATA_CODE_PATTERN))
airport_rows_after = valid_airport_df.count()
print(
    f"By-airport aggregation: excluded {airport_rows_before - airport_rows_after} rows "
    f"with non-IATA ORIGIN_AIRPORT codes (kept {airport_rows_after} of {airport_rows_before})"
)

by_airport = (
    delay_rate_agg(valid_airport_df, ["ORIGIN_AIRPORT"])
    .filter(F.col("total_flights") > 10000)
    .orderBy(F.desc("total_flights"))
)
save(by_airport, "agg_by_airport_v2")

# c. By SCHEDULED_DEPARTURE_HOUR
by_hour = delay_rate_agg(df, ["SCHEDULED_DEPARTURE_HOUR"]).orderBy(F.asc("SCHEDULED_DEPARTURE_HOUR"))
save(by_hour, "agg_by_hour_v2")

# d. By DAY_OF_WEEK
by_dow = delay_rate_agg(df, ["DAY_OF_WEEK"]).orderBy(F.asc("DAY_OF_WEEK"))
save(by_dow, "agg_by_day_of_week_v2")

# e. By MONTH
by_month = delay_rate_agg(df, ["MONTH"]).orderBy(F.asc("MONTH"))
save(by_month, "agg_by_month_v2")

# Overall delay-cause totals (minutes) -> pie chart in the EDA section
cause_sums = df.agg(*[F.sum(c).alias(c) for c in DELAY_CAUSE_COLS_AGG]).collect()[0]
cause_totals = {c: float(cause_sums[c] or 0.0) for c in DELAY_CAUSE_COLS_AGG}
with open(f"{METRICS_DIR}/delay_cause_totals_v2.json", "w") as f:
    json.dump(cause_totals, f, indent=2)

print("\n(a) Delay rate by airline (top 10):")
by_airline.show(10, truncate=False)
print("(b) Delay rate by origin airport, total_flights > 10000 (top 10 busiest):")
by_airport.show(10, truncate=False)
print("(c) Delay rate by scheduled departure hour (top 10):")
by_hour.show(10, truncate=False)
print("(d) Delay rate by day of week:")
by_dow.show(10, truncate=False)
print("(e) Delay rate by month:")
by_month.show(12, truncate=False)
print("Overall delay-cause totals (minutes):")
print(json.dumps(cause_totals, indent=2))

total_rows_agg = df.count()
overall_delay_pct = round(100.0 * df.filter(F.col("is_delayed") == 1).count() / total_rows_agg, 2)
print(f"\nOverall delay percentage: {overall_delay_pct}% (sanity check vs 18.02% from the clean step)")

## 6. Exploratory Data Analysis — 6 charts (2015+2016 combined)

Same six charts as the single-year baseline, rendered inline and saved to
`/content/outputs/charts/` as `*_v2.png`.

In [ ]:
%matplotlib inline
import glob

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

CHARTS_DIR = "/content/outputs/charts"
os.makedirs(CHARTS_DIR, exist_ok=True)
sns.set_style("whitegrid")
DPI = 150

MONTH_LABELS = {1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr", 5: "May", 6: "Jun",
                7: "Jul", 8: "Aug", 9: "Sep", 10: "Oct", 11: "Nov", 12: "Dec"}
DOW_LABELS = {1: "Mon", 2: "Tue", 3: "Wed", 4: "Thu", 5: "Fri", 6: "Sat", 7: "Sun"}


def read_agg_csv(name):
    path = glob.glob(f"{PROCESSED_DIR}/{name}.csv/part-*.csv")[0]
    return pd.read_csv(path)


findings = []

by_airline_pd = read_agg_csv("agg_by_airline_v2").sort_values("delay_pct", ascending=False).head(10)
fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(data=by_airline_pd.sort_values("delay_pct"), x="delay_pct", y="AIRLINE_NAME", color="#4C72B0", ax=ax)
ax.set_title("Top 10 Worst Airlines by Delay Rate (2015+2016)")
ax.set_xlabel("Delay Rate (%)")
ax.set_ylabel("Airline")
fig.tight_layout()
fig.savefig(f"{CHARTS_DIR}/01_delay_by_airline_v2.png", dpi=DPI)
plt.show()

worst = by_airline_pd.iloc[0]
best_of_worst = by_airline_pd.iloc[-1]
findings.append(
    f"Chart 1 (delay by airline): {worst['AIRLINE_NAME']} has the worst delay rate at "
    f"{worst['delay_pct']}%, roughly {worst['delay_pct'] / best_of_worst['delay_pct']:.1f}x the "
    f"rate of {best_of_worst['AIRLINE_NAME']} ({best_of_worst['delay_pct']}%)."
)

by_airport_pd = read_agg_csv("agg_by_airport_v2").sort_values("total_flights", ascending=False).head(10)
fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(data=by_airport_pd.sort_values("total_flights"), x="delay_pct", y="ORIGIN_AIRPORT", color="#DD8452", ax=ax)
ax.set_title("Delay Rate at the 10 Busiest Origin Airports (2015+2016)")
ax.set_xlabel("Delay Rate (%)")
ax.set_ylabel("Airport (IATA code)")
fig.tight_layout()
fig.savefig(f"{CHARTS_DIR}/02_delay_by_airport_v2.png", dpi=DPI)
plt.show()

worst_airport = by_airport_pd.loc[by_airport_pd["delay_pct"].idxmax()]
best_airport = by_airport_pd.loc[by_airport_pd["delay_pct"].idxmin()]
findings.append(
    f"Chart 2 (delay by airport): {worst_airport['ORIGIN_AIRPORT']} has the highest delay rate "
    f"({worst_airport['delay_pct']}%) among the busiest 10, {best_airport['ORIGIN_AIRPORT']} the "
    f"lowest ({best_airport['delay_pct']}%)."
)

with open(f"{METRICS_DIR}/delay_cause_totals_v2.json") as f:
    cause_totals = json.load(f)
labels = list(cause_totals.keys())
values = list(cause_totals.values())
fig, ax = plt.subplots(figsize=(7, 7))
ax.pie(values, labels=labels, autopct="%1.1f%%", startangle=90, colors=sns.color_palette("Set2", len(values)))
ax.set_title("Total Delay Minutes by Cause (2015+2016)")
fig.tight_layout()
fig.savefig(f"{CHARTS_DIR}/03_delay_cause_breakdown_v2.png", dpi=DPI)
plt.show()

total_minutes = sum(values)
top_cause = max(cause_totals, key=cause_totals.get)
top_cause_pct = 100.0 * cause_totals[top_cause] / total_minutes
findings.append(
    f"Chart 3 (delay cause breakdown): {top_cause.replace('_', ' ').title()} is the largest "
    f"contributor at {top_cause_pct:.1f}% of total delay minutes."
)

by_hour_pd = read_agg_csv("agg_by_hour_v2").sort_values("SCHEDULED_DEPARTURE_HOUR")
fig, ax = plt.subplots(figsize=(10, 6))
sns.lineplot(data=by_hour_pd, x="SCHEDULED_DEPARTURE_HOUR", y="delay_pct", marker="o", ax=ax)
ax.set_title("Delay Rate by Scheduled Departure Hour (2015+2016)")
ax.set_xlabel("Scheduled Departure Hour (0-23)")
ax.set_ylabel("Delay Rate (%)")
ax.set_xticks(range(0, 24))
fig.tight_layout()
fig.savefig(f"{CHARTS_DIR}/04_delay_by_hour_v2.png", dpi=DPI)
plt.show()

morning = by_hour_pd[by_hour_pd["SCHEDULED_DEPARTURE_HOUR"].between(5, 8)]["delay_pct"].mean()
evening = by_hour_pd[by_hour_pd["SCHEDULED_DEPARTURE_HOUR"].between(18, 21)]["delay_pct"].mean()
findings.append(
    f"Chart 4 (delay by hour): morning (5-8am) averages {morning:.1f}% vs evening (6-9pm) "
    f"{evening:.1f}%, roughly {evening / morning:.1f}x higher."
)

by_dow_pd = read_agg_csv("agg_by_day_of_week_v2").sort_values("DAY_OF_WEEK")
by_dow_pd["DOW_LABEL"] = by_dow_pd["DAY_OF_WEEK"].map(DOW_LABELS)
fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(data=by_dow_pd, x="DOW_LABEL", y="delay_pct", color="#55A868", ax=ax)
ax.set_title("Delay Rate by Day of Week (2015+2016)")
ax.set_xlabel("Day of Week")
ax.set_ylabel("Delay Rate (%)")
fig.tight_layout()
fig.savefig(f"{CHARTS_DIR}/05_delay_by_dayofweek_v2.png", dpi=DPI)
plt.show()

worst_dow = by_dow_pd.loc[by_dow_pd["delay_pct"].idxmax()]
best_dow = by_dow_pd.loc[by_dow_pd["delay_pct"].idxmin()]
findings.append(
    f"Chart 5 (delay by day of week): {worst_dow['DOW_LABEL']} worst ({worst_dow['delay_pct']}%), "
    f"{best_dow['DOW_LABEL']} best ({best_dow['delay_pct']}%)."
)

by_month_pd = read_agg_csv("agg_by_month_v2").sort_values("MONTH")
by_month_pd["MONTH_LABEL"] = by_month_pd["MONTH"].map(MONTH_LABELS)
fig, ax = plt.subplots(figsize=(10, 6))
sns.lineplot(data=by_month_pd, x="MONTH", y="delay_pct", marker="o", ax=ax)
ax.set_title("Delay Rate by Month (2015+2016)")
ax.set_xlabel("Month")
ax.set_ylabel("Delay Rate (%)")
ax.set_xticks(by_month_pd["MONTH"])
ax.set_xticklabels(by_month_pd["MONTH_LABEL"])
fig.tight_layout()
fig.savefig(f"{CHARTS_DIR}/06_delay_by_month_v2.png", dpi=DPI)
plt.show()

worst_month = by_month_pd.loc[by_month_pd["delay_pct"].idxmax()]
best_month = by_month_pd.loc[by_month_pd["delay_pct"].idxmin()]
findings.append(
    f"Chart 6 (delay by month): {worst_month['MONTH_LABEL']} worst ({worst_month['delay_pct']}%), "
    f"{best_month['MONTH_LABEL']} best ({best_month['delay_pct']}%)."
)

print("=== Findings summary (2015+2016 combined) ===")
for line in findings:
    print(f"- {line}")

## 7. Model — predicting delays from pre-flight information (2015+2016 combined)

Same six pre-flight-known features, same leakage exclusions, same class
imbalance (18.02% delayed — barely different from the single-year 18.61%) and
the same fix: unweighted baseline first (to show the majority-class
degeneracy concretely), then inverse-frequency class weighting plus a
threshold sweep tuned for the delayed class's F1 score.

In [ ]:
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.functions import vector_to_array

LABEL_COL = "is_delayed"
FEATURE_COLS = [
    "AIRLINE_INDEXED", "ORIGIN_AIRPORT_INDEXED", "MONTH",
    "DAY_OF_WEEK", "SCHEDULED_DEPARTURE_HOUR", "DISTANCE",
]

model_df = spark.read.parquet(f"{PROCESSED_DIR}/flights_clean_v2.parquet").select(
    "AIRLINE", "ORIGIN_AIRPORT", "MONTH", "DAY_OF_WEEK",
    "SCHEDULED_DEPARTURE_HOUR", "DISTANCE", LABEL_COL,
)
model_df.persist(StorageLevel.DISK_ONLY)
print(f"Total rows for modeling (2015+2016 combined): {model_df.count()}")

airline_indexer = StringIndexer(inputCol="AIRLINE", outputCol="AIRLINE_INDEXED", handleInvalid="keep")
airport_indexer = StringIndexer(inputCol="ORIGIN_AIRPORT", outputCol="ORIGIN_AIRPORT_INDEXED", handleInvalid="keep")
model_df = airline_indexer.fit(model_df).transform(model_df)
model_df = airport_indexer.fit(model_df).transform(model_df)

assembler = VectorAssembler(inputCols=FEATURE_COLS, outputCol="features")
model_df = assembler.transform(model_df)

train_df, test_df = model_df.randomSplit([0.8, 0.2], seed=42)
train_df.persist(StorageLevel.DISK_ONLY)
test_df.persist(StorageLevel.DISK_ONLY)
print(f"Train rows: {train_df.count()}, Test rows: {test_df.count()}")

test_on_time = test_df.filter(f"{LABEL_COL} == 0").count()
majority_baseline_accuracy = round(test_on_time / test_df.count(), 4)
print(f"\n'Always predict on-time' baseline accuracy on test set: {majority_baseline_accuracy}")


def evaluate(predictions, model_name):
    auc = BinaryClassificationEvaluator(
        labelCol=LABEL_COL, rawPredictionCol="rawPrediction", metricName="areaUnderROC"
    ).evaluate(predictions)
    metrics = {
        m: MulticlassClassificationEvaluator(labelCol=LABEL_COL, predictionCol="prediction", metricName=m).evaluate(predictions)
        for m in ["accuracy", "f1", "weightedPrecision", "weightedRecall"]
    }
    confusion = predictions.groupBy(LABEL_COL, "prediction").count().orderBy(LABEL_COL, "prediction").collect()
    confusion_counts = {f"actual_{int(r[LABEL_COL])}_predicted_{int(r['prediction'])}": r["count"] for r in confusion}
    result = {
        "model": model_name, "auc": round(auc, 4), "accuracy": round(metrics["accuracy"], 4),
        "f1": round(metrics["f1"], 4), "precision": round(metrics["weightedPrecision"], 4),
        "recall": round(metrics["weightedRecall"], 4), "confusion_matrix": confusion_counts,
    }
    print(f"\n=== {model_name} ===")
    for k in ["auc", "accuracy", "f1", "precision", "recall"]:
        print(f"{k}: {result[k]}")
    print(f"Confusion matrix: {confusion_counts}")
    return result


# --- Baseline (unweighted) models — expected to degenerate to majority-class ---
lr = LogisticRegression(featuresCol="features", labelCol=LABEL_COL)
lr_result = evaluate(lr.fit(train_df).transform(test_df), "LogisticRegression (unweighted)")

rf = RandomForestClassifier(
    featuresCol="features", labelCol=LABEL_COL, numTrees=50, maxDepth=8, maxBins=700, seed=42,
)  # maxBins must be >= ORIGIN_AIRPORT's distinct indexed values
rf_model = rf.fit(train_df)
rf_result = evaluate(rf_model.transform(test_df), "RandomForest (unweighted)")

print(
    "\n-> As expected: both unweighted models are functionally identical to the "
    f"majority-class baseline ({majority_baseline_accuracy * 100:.2f}% accuracy) — "
    "0 true positives for the delayed class."
)

### Fixing the imbalance: class weighting + threshold tuning

In [ ]:
THRESHOLDS = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]


def sweep_thresholds(predictions, model_name):
    predictions = predictions.withColumn("prob_delayed", vector_to_array("probability")[1])
    predictions.persist(StorageLevel.DISK_ONLY)
    predictions.count()

    print(f"\n{model_name} — threshold sweep (precision/recall/F1 for delayed class):")
    print(f"{'threshold':>10} {'precision':>10} {'recall':>10} {'f1':>10}")

    sweep_results = []
    for t in THRESHOLDS:
        row = predictions.agg(
            F.sum(F.when((F.col(LABEL_COL) == 1) & (F.col("prob_delayed") >= t), 1).otherwise(0)).alias("tp"),
            F.sum(F.when((F.col(LABEL_COL) == 0) & (F.col("prob_delayed") >= t), 1).otherwise(0)).alias("fp"),
            F.sum(F.when((F.col(LABEL_COL) == 1) & (F.col("prob_delayed") < t), 1).otherwise(0)).alias("fn"),
            F.sum(F.when((F.col(LABEL_COL) == 0) & (F.col("prob_delayed") < t), 1).otherwise(0)).alias("tn"),
        ).collect()[0]
        tp, fp, fn = row["tp"], row["fp"], row["fn"]
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        sweep_results.append({"threshold": t, "precision": round(precision, 4), "recall": round(recall, 4), "f1": round(f1, 4)})
        print(f"{t:>10.2f} {precision:>10.4f} {recall:>10.4f} {f1:>10.4f}")

    predictions.unpersist()
    best = max(sweep_results, key=lambda r: r["f1"])
    print(f"Best threshold for {model_name} (max F1): {best['threshold']} -> {best}")
    return sweep_results, best


label_counts = {int(r[LABEL_COL]): r["count"] for r in train_df.groupBy(LABEL_COL).count().collect()}
total_train = sum(label_counts.values())
class_weights = {label: total_train / (2 * count) for label, count in label_counts.items()}
print(f"Class weights (total / (2 * class_count)): {class_weights}")

train_weighted = train_df.withColumn(
    "classWeight", F.when(F.col(LABEL_COL) == 1, class_weights[1]).otherwise(class_weights[0])
)
train_weighted.persist(StorageLevel.DISK_ONLY)
train_weighted.count()

lr_weighted = LogisticRegression(featuresCol="features", labelCol=LABEL_COL, weightCol="classWeight")
lr_weighted_predictions = lr_weighted.fit(train_weighted).transform(test_df)
print(f"\nWeighted LogisticRegression AUC: {round(BinaryClassificationEvaluator(labelCol=LABEL_COL).evaluate(lr_weighted_predictions), 4)}")
lr_sweep, lr_best = sweep_thresholds(lr_weighted_predictions, "Weighted LogisticRegression")

rf_weighted = RandomForestClassifier(
    featuresCol="features", labelCol=LABEL_COL, numTrees=50, maxDepth=8,
    maxBins=700, seed=42, weightCol="classWeight",
)
rf_weighted_model = rf_weighted.fit(train_weighted)
rf_weighted_predictions = rf_weighted_model.transform(test_df)
rf_weighted_auc = BinaryClassificationEvaluator(labelCol=LABEL_COL).evaluate(rf_weighted_predictions)
print(f"\nWeighted RandomForest AUC: {round(rf_weighted_auc, 4)}")
rf_sweep, rf_best = sweep_thresholds(rf_weighted_predictions, "Weighted RandomForest")

# Confusion matrix for the weighted RF model at its best threshold
rf_best_t = rf_best["threshold"]
rf_tuned = rf_weighted_predictions.withColumn(
    "prob_delayed", vector_to_array("probability")[1]
).withColumn("tuned_prediction", F.when(F.col("prob_delayed") >= rf_best_t, 1).otherwise(0))
rf_tuned_confusion = rf_tuned.groupBy(LABEL_COL, "tuned_prediction").count().orderBy(LABEL_COL, "tuned_prediction").collect()
rf_tuned_confusion_counts = {f"actual_{int(r[LABEL_COL])}_predicted_{int(r['tuned_prediction'])}": r["count"] for r in rf_tuned_confusion}

print("\n=== Before vs After (2015+2016 combined) ===")
print(f"BEFORE (unweighted RF, 0.5 threshold): {rf_result['confusion_matrix']} -> 0 true positives, identical to majority baseline")
print(f"AFTER  (weighted RF, tuned threshold={rf_best_t}): precision={rf_best['precision']}, recall={rf_best['recall']}, f1={rf_best['f1']}, confusion={rf_tuned_confusion_counts}")

feature_importances = sorted(zip(FEATURE_COLS, rf_weighted_model.featureImportances.toArray()), key=lambda x: x[1], reverse=True)
print("\nWeighted RandomForest feature importances (sorted descending):")
for feature, importance in feature_importances:
    print(f"  {feature}: {round(float(importance), 4)}")

model_metrics_v2 = {
    "majority_class_baseline_accuracy": majority_baseline_accuracy,
    "unweighted_logistic_regression": lr_result,
    "unweighted_random_forest": rf_result,
    "weighted_logistic_regression": {"auc": round(BinaryClassificationEvaluator(labelCol=LABEL_COL).evaluate(lr_weighted_predictions), 4), "threshold_sweep": lr_sweep, "best_threshold": lr_best},
    "weighted_random_forest": {"auc": round(rf_weighted_auc, 4), "threshold_sweep": rf_sweep, "best_threshold": rf_best, "confusion_matrix_at_best_threshold": rf_tuned_confusion_counts},
}
with open(f"{METRICS_DIR}/model_metrics_2yr_notebook.json", "w") as f:
    json.dump(model_metrics_v2, f, indent=2)
print(f"\nMetrics saved to {METRICS_DIR}/model_metrics_2yr_notebook.json")

### Doubling the data barely moved model performance — a real finding

The single-year baseline (Appendix) got weighted-RF **AUC 0.649**
(precision 0.258, recall 0.637). The combined 2015+2016 run above gets
**AUC 0.649** (precision 0.252, recall 0.629) — essentially unchanged,
despite training on roughly double the rows (11.25M vs 5.7M). This is a
genuine, useful negative result: **the model's limitation is feature
richness, not data volume.** Six coarse features (airline, origin airport,
month, day-of-week, scheduled hour, distance) only carry so much predictive
signal about a fundamentally noisy, cascading, operationally-driven
phenomenon (recall from the EDA: Late Aircraft Delay is the single largest
delay-minutes contributor, i.e. much of the noise is knock-on effects from
*other* flights, not something encoded in this flight's own static
attributes). Simply feeding the model more rows of the same six columns
was never going to fix that — the next lever for a stronger model is richer
features (e.g. weather at origin/destination, upstream aircraft/tail-number
delay chains, airport congestion at scheduled time), not more data.

## 8. Conclusion

**Key findings from the EDA (11.25M cleaned flights, 2015+2016 combined,
18.02% delayed overall — nearly identical to the single-year 18.61%):**

- **Worst airline:** Spirit Air Lines, 26.69% delay rate — roughly 1.4x the
  rate of Southwest Airlines Co. (18.52%), the best-performing airline among
  the worst-10.
- **Worst busy airport (top-10 busiest):** ORD (Chicago O'Hare), 22.65% —
  while ATL, the single busiest airport in the combined data (757,318
  flights), has a comparatively low 15.23%. Busiest does not mean most
  delayed.
- **Top delay cause:** Late Aircraft Delay, accounting for ~39.5% of all
  delay minutes — consistent with the single-year finding, more than double
  Weather Delay's share (~5.4%). Most delay time is operational knock-on
  effects, not weather.
- **Time-of-day pattern:** delay rate climbs steadily through the day —
  early-morning flights (5-8am) average ~9.7% delayed versus ~25.1% for
  evening flights (6-9pm), roughly 2.6x higher.
- **Day-of-week pattern:** Thursday is the worst day to fly (19.76%),
  Saturday the best (15.54%).
- **Month pattern:** June is the worst month (22.19%), October the best
  (13.00%).
- **Data quality — a genuine cross-source comparison:** the original
  `usdot/flight-delays` source has ~8.4% of rows (all of October 2015) using
  numeric DOT airport IDs instead of IATA codes. Running the identical
  defensive filter against this combined source (`yuanyuwendymu/...`) found
  **0 rows to exclude** — same real-world flights, different repackaging,
  different data quality quirks. A reminder that raw data always needs
  verifying rather than assuming consistency across sources claiming to
  cover the same events.

**Model performance, honestly reported:** exactly as with the single-year
baseline, both unweighted Logistic Regression and Random Forest degenerate
to the majority-class baseline (81.96% accuracy, 0 true positives for the
delayed class) when trained without addressing the class imbalance.
Inverse-frequency class weighting plus a threshold sweep (tuned for the
delayed class's F1 score, best threshold = 0.5) produces the same honest
trade-off as before: the tuned weighted Random Forest catches **~63% of
real delays (recall)**, but only **~25% of its "delayed" predictions are
correct (precision)** — roughly three out of four delay alarms are false
positives. **Critically, this is barely different from the single-year
model's AUC of 0.649** — doubling the training data from 5.7M to 11.25M
rows did not meaningfully improve separability. That's a real, useful
finding for the report: the bottleneck is feature richness (six coarse
pre-flight columns), not data volume, and any downstream use (e.g. the
project's dashboard) should keep presenting this as a *screening* signal,
not a high-confidence prediction.

---
# Appendix: Single-Year Baseline (2015 only, original pipeline)

This appendix preserves the project's **original** single-year run
(`usdot/flight-delays`, `flights.csv`, 5,819,079 raw rows / 5,714,008
cleaned rows) exactly as first built and verified, before the project scaled
up to the combined 2015+2016 dataset above. It reuses the `airlines.csv` /
`airports.csv` lookup tables and `flights.csv` already downloaded in Section 2,
and the same `spark` session (already running above) — no new downloads or
setup needed.

**Why keep this:** it's a clean before/after scaling narrative — the project
started with a single year to validate the approach end-to-end, then scaled
to a combined multi-year dataset to meet a GB-scale requirement, and (per the
Conclusion above) the findings and model performance held essentially
unchanged. That stability is itself evidence the original single-year
pipeline was already sound, not just lucky.

## Appendix A.1 — Ingest (2015 only) — explicit schema, join lookup tables

`flights.csv` is ~5.8M rows. Letting Spark run `inferSchema` on a file this
size means an extra full pass just to guess column types before the real
read even starts — wasteful at this scale, so instead we define an explicit
`StructType` up front (this mirrors `src/ingest.py` and the project's own
"Do NOT run schema inference on the full 5.8M row CSV" rule).

We then left-join:
- `flights.AIRLINE` (IATA code) → `airlines.csv` → full airline name
- `flights.ORIGIN_AIRPORT` (IATA code) → `airports.csv` → origin city/state

The result is a wide, **joined-but-unfiltered** dataframe — no rows are
dropped yet, that happens in the cleaning step next.

In [ ]:
FLIGHTS_SCHEMA = StructType(
    [
        StructField("YEAR", IntegerType(), True),
        StructField("MONTH", IntegerType(), True),
        StructField("DAY", IntegerType(), True),
        StructField("DAY_OF_WEEK", IntegerType(), True),
        StructField("AIRLINE", StringType(), True),
        StructField("FLIGHT_NUMBER", IntegerType(), True),
        StructField("TAIL_NUMBER", StringType(), True),
        StructField("ORIGIN_AIRPORT", StringType(), True),
        StructField("DESTINATION_AIRPORT", StringType(), True),
        StructField("SCHEDULED_DEPARTURE", IntegerType(), True),
        StructField("DEPARTURE_TIME", IntegerType(), True),
        StructField("DEPARTURE_DELAY", DoubleType(), True),
        StructField("TAXI_OUT", DoubleType(), True),
        StructField("WHEELS_OFF", IntegerType(), True),
        StructField("SCHEDULED_TIME", DoubleType(), True),
        StructField("ELAPSED_TIME", DoubleType(), True),
        StructField("AIR_TIME", DoubleType(), True),
        StructField("DISTANCE", IntegerType(), True),
        StructField("WHEELS_ON", IntegerType(), True),
        StructField("TAXI_IN", DoubleType(), True),
        StructField("SCHEDULED_ARRIVAL", IntegerType(), True),
        StructField("ARRIVAL_TIME", IntegerType(), True),
        StructField("ARRIVAL_DELAY", DoubleType(), True),
        StructField("DIVERTED", IntegerType(), True),
        StructField("CANCELLED", IntegerType(), True),
        StructField("CANCELLATION_REASON", StringType(), True),
        StructField("AIR_SYSTEM_DELAY", DoubleType(), True),
        StructField("SECURITY_DELAY", DoubleType(), True),
        StructField("AIRLINE_DELAY", DoubleType(), True),
        StructField("LATE_AIRCRAFT_DELAY", DoubleType(), True),
        StructField("WEATHER_DELAY", DoubleType(), True),
    ]
)

flights = spark.read.csv(f"{RAW_DIR}/flights.csv", header=True, schema=FLIGHTS_SCHEMA)
# airlines / airports already read+schema'd in Section 3 above; reuse them here.

airlines_renamed_appendix = airlines.withColumnRenamed("AIRLINE", "AIRLINE_NAME")
joined_appendix = flights.join(
    airlines_renamed_appendix, flights["AIRLINE"] == airlines_renamed_appendix["IATA_CODE"], "left"
).drop(airlines_renamed_appendix["IATA_CODE"])

origin_airports_appendix = (
    airports.withColumnRenamed("CITY", "ORIGIN_CITY")
    .withColumnRenamed("STATE", "ORIGIN_STATE")
    .select("IATA_CODE", "ORIGIN_CITY", "ORIGIN_STATE")
)
joined_appendix = joined_appendix.join(
    origin_airports_appendix, joined_appendix["ORIGIN_AIRPORT"] == origin_airports_appendix["IATA_CODE"], "left"
).drop(origin_airports_appendix["IATA_CODE"])

row_count = joined_appendix.count()
print(f"\nJoined row count (2015 only): {row_count}\n")
print("Joined schema:")
joined_appendix.printSchema()

joined_appendix.write.mode("overwrite").parquet(f"{PROCESSED_DIR}/flights_joined.parquet")

print("\nSample rows:")
joined_appendix.select(
    "YEAR", "MONTH", "DAY", "AIRLINE", "AIRLINE_NAME",
    "ORIGIN_AIRPORT", "ORIGIN_CITY", "ORIGIN_STATE",
    "DESTINATION_AIRPORT", "ARRIVAL_DELAY",
).show(5, truncate=False)

## Appendix A.2 — Clean (2015 only) — filter invalid flights, build the `is_delayed` label

Same logic as Section 4 above: drop `CANCELLED`/`DIVERTED` flights, label
`is_delayed = 1` if `ARRIVAL_DELAY >= 15` minutes (FAA standard), fill the
five `*_DELAY` cause columns' nulls with `0`, derive
`SCHEDULED_DEPARTURE_HOUR`.

In [ ]:
df_appendix = spark.read.parquet(f"{PROCESSED_DIR}/flights_joined.parquet")

total_rows = df_appendix.count()
print(f"\nStarting row count: {total_rows}")

cancelled_count = df_appendix.filter(F.col("CANCELLED") == 1).count()
diverted_count = df_appendix.filter((F.col("CANCELLED") != 1) & (F.col("DIVERTED") == 1)).count()
print(f"Rows dropped for CANCELLED == 1: {cancelled_count}")
print(f"Rows dropped for DIVERTED == 1 (excluding already-cancelled): {diverted_count}")

df_appendix = df_appendix.filter((F.col("CANCELLED") != 1) & (F.col("DIVERTED") != 1))
after_filter_rows = df_appendix.count()
print(f"Row count after dropping cancelled/diverted: {after_filter_rows}")

df_appendix = df_appendix.fillna(0, subset=DELAY_CAUSE_COLUMNS)

df_appendix = df_appendix.withColumn("is_delayed", F.when(F.col("ARRIVAL_DELAY") >= 15, 1).otherwise(0))

delayed_count = df_appendix.filter(F.col("is_delayed") == 1).count()
on_time_count = after_filter_rows - delayed_count
pct_delayed = 100.0 * delayed_count / after_filter_rows
pct_on_time = 100.0 * on_time_count / after_filter_rows
print(
    f"\nClass balance — delayed: {delayed_count} ({pct_delayed:.2f}%), "
    f"on-time: {on_time_count} ({pct_on_time:.2f}%)"
)

df_appendix = df_appendix.withColumn(
    "SCHEDULED_DEPARTURE_HOUR", (F.col("SCHEDULED_DEPARTURE") / 100).cast("int")
)

final_row_count = df_appendix.count()
print(f"\nFinal row count: {final_row_count}")

df_appendix.write.mode("overwrite").parquet(f"{PROCESSED_DIR}/flights_clean.parquet")

print("\nSample rows:")
df_appendix.select(
    "YEAR", "MONTH", "DAY", "AIRLINE", "SCHEDULED_DEPARTURE",
    "SCHEDULED_DEPARTURE_HOUR", "ARRIVAL_DELAY", "is_delayed",
    "AIR_SYSTEM_DELAY", "SECURITY_DELAY", "AIRLINE_DELAY",
    "LATE_AIRCRAFT_DELAY", "WEATHER_DELAY",
).show(5, truncate=False)

## Appendix A.3 — Aggregate (2015 only)

Same aggregation logic as Section 5 above.

> **Data quality finding (original discovery):** while building the
> by-airport aggregation on this source, we discovered that **all of October
> 2015** (~8.4% of rows, 482,878 flights) uses **numeric DOT airport IDs
> instead of 3-letter IATA codes** in `ORIGIN_AIRPORT`/`DESTINATION_AIRPORT`
> — a known quirk of the `usdot/flight-delays` dataset specifically (see
> Section 5 above for the comparison against the combined source, which does
> *not* have this issue). The fix: filter `ORIGIN_AIRPORT` to `^[A-Z]{3}$`
> only for the airport-keyed aggregation.

In [ ]:
df_appendix.persist(StorageLevel.DISK_ONLY)

# a. By AIRLINE_NAME
by_airline_appendix = delay_rate_agg(df_appendix, ["AIRLINE_NAME"]).orderBy(F.desc("delay_pct"))
save(by_airline_appendix, "agg_by_airline")

# b. By ORIGIN_AIRPORT, IATA-only + busy airports
airport_rows_before = df_appendix.select("ORIGIN_AIRPORT").count()
valid_airport_df_appendix = df_appendix.filter(F.col("ORIGIN_AIRPORT").rlike(IATA_CODE_PATTERN))
airport_rows_after = valid_airport_df_appendix.count()
print(
    f"By-airport aggregation: excluded {airport_rows_before - airport_rows_after} rows "
    f"with non-IATA ORIGIN_AIRPORT codes (kept {airport_rows_after} of {airport_rows_before})"
)

by_airport_appendix = (
    delay_rate_agg(valid_airport_df_appendix, ["ORIGIN_AIRPORT"])
    .filter(F.col("total_flights") > 10000)
    .orderBy(F.desc("total_flights"))
)
save(by_airport_appendix, "agg_by_airport")

# c. By SCHEDULED_DEPARTURE_HOUR
by_hour_appendix = delay_rate_agg(df_appendix, ["SCHEDULED_DEPARTURE_HOUR"]).orderBy(F.asc("SCHEDULED_DEPARTURE_HOUR"))
save(by_hour_appendix, "agg_by_hour")

# d. By DAY_OF_WEEK
by_dow_appendix = delay_rate_agg(df_appendix, ["DAY_OF_WEEK"]).orderBy(F.asc("DAY_OF_WEEK"))
save(by_dow_appendix, "agg_by_day_of_week")

# e. By MONTH
by_month_appendix = delay_rate_agg(df_appendix, ["MONTH"]).orderBy(F.asc("MONTH"))
save(by_month_appendix, "agg_by_month")

# Overall delay-cause totals (minutes)
cause_sums_appendix = df_appendix.agg(*[F.sum(c).alias(c) for c in DELAY_CAUSE_COLS_AGG]).collect()[0]
cause_totals_appendix = {c: float(cause_sums_appendix[c] or 0.0) for c in DELAY_CAUSE_COLS_AGG}
with open(f"{METRICS_DIR}/delay_cause_totals.json", "w") as f:
    json.dump(cause_totals_appendix, f, indent=2)

print("\n(a) Delay rate by airline (top 10):")
by_airline_appendix.show(10, truncate=False)
print("(b) Delay rate by origin airport, total_flights > 10000 (top 10 busiest):")
by_airport_appendix.show(10, truncate=False)
print("(c) Delay rate by scheduled departure hour (top 10):")
by_hour_appendix.show(10, truncate=False)
print("(d) Delay rate by day of week:")
by_dow_appendix.show(10, truncate=False)
print("(e) Delay rate by month (top 10):")
by_month_appendix.show(10, truncate=False)
print("Overall delay-cause totals (minutes):")
print(json.dumps(cause_totals_appendix, indent=2))

total_rows_agg = df_appendix.count()
overall_delay_pct = round(100.0 * df_appendix.filter(F.col("is_delayed") == 1).count() / total_rows_agg, 2)
print(f"\nOverall delay percentage: {overall_delay_pct}% (sanity check vs 18.61% from the clean step)")

## Appendix A.4 — Exploratory Data Analysis (2015 only) — 6 charts

In [ ]:
by_airline_pd_appendix = read_agg_csv("agg_by_airline").sort_values("delay_pct", ascending=False).head(10)
fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(data=by_airline_pd_appendix.sort_values("delay_pct"), x="delay_pct", y="AIRLINE_NAME", color="#4C72B0", ax=ax)
ax.set_title("Top 10 Worst Airlines by Delay Rate (2015 only)")
ax.set_xlabel("Delay Rate (%)")
ax.set_ylabel("Airline")
fig.tight_layout()
fig.savefig(f"{CHARTS_DIR}/01_delay_by_airline.png", dpi=DPI)
plt.show()

by_airport_pd_appendix = read_agg_csv("agg_by_airport").sort_values("total_flights", ascending=False).head(10)
fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(data=by_airport_pd_appendix.sort_values("total_flights"), x="delay_pct", y="ORIGIN_AIRPORT", color="#DD8452", ax=ax)
ax.set_title("Delay Rate at the 10 Busiest Origin Airports (2015 only)")
ax.set_xlabel("Delay Rate (%)")
ax.set_ylabel("Airport (IATA code)")
fig.tight_layout()
fig.savefig(f"{CHARTS_DIR}/02_delay_by_airport.png", dpi=DPI)
plt.show()

with open(f"{METRICS_DIR}/delay_cause_totals.json") as f:
    cause_totals_appendix = json.load(f)
labels = list(cause_totals_appendix.keys())
values = list(cause_totals_appendix.values())
fig, ax = plt.subplots(figsize=(7, 7))
ax.pie(values, labels=labels, autopct="%1.1f%%", startangle=90, colors=sns.color_palette("Set2", len(values)))
ax.set_title("Total Delay Minutes by Cause (2015 only)")
fig.tight_layout()
fig.savefig(f"{CHARTS_DIR}/03_delay_cause_breakdown.png", dpi=DPI)
plt.show()

by_hour_pd_appendix = read_agg_csv("agg_by_hour").sort_values("SCHEDULED_DEPARTURE_HOUR")
fig, ax = plt.subplots(figsize=(10, 6))
sns.lineplot(data=by_hour_pd_appendix, x="SCHEDULED_DEPARTURE_HOUR", y="delay_pct", marker="o", ax=ax)
ax.set_title("Delay Rate by Scheduled Departure Hour (2015 only)")
ax.set_xlabel("Scheduled Departure Hour (0-23)")
ax.set_ylabel("Delay Rate (%)")
ax.set_xticks(range(0, 24))
fig.tight_layout()
fig.savefig(f"{CHARTS_DIR}/04_delay_by_hour.png", dpi=DPI)
plt.show()

by_dow_pd_appendix = read_agg_csv("agg_by_day_of_week").sort_values("DAY_OF_WEEK")
by_dow_pd_appendix["DOW_LABEL"] = by_dow_pd_appendix["DAY_OF_WEEK"].map(DOW_LABELS)
fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(data=by_dow_pd_appendix, x="DOW_LABEL", y="delay_pct", color="#55A868", ax=ax)
ax.set_title("Delay Rate by Day of Week (2015 only)")
ax.set_xlabel("Day of Week")
ax.set_ylabel("Delay Rate (%)")
fig.tight_layout()
fig.savefig(f"{CHARTS_DIR}/05_delay_by_dayofweek.png", dpi=DPI)
plt.show()

by_month_pd_appendix = read_agg_csv("agg_by_month").sort_values("MONTH")
by_month_pd_appendix["MONTH_LABEL"] = by_month_pd_appendix["MONTH"].map(MONTH_LABELS)
fig, ax = plt.subplots(figsize=(10, 6))
sns.lineplot(data=by_month_pd_appendix, x="MONTH", y="delay_pct", marker="o", ax=ax)
ax.set_title("Delay Rate by Month (2015 only)")
ax.set_xlabel("Month")
ax.set_ylabel("Delay Rate (%)")
ax.set_xticks(by_month_pd_appendix["MONTH"])
ax.set_xticklabels(by_month_pd_appendix["MONTH_LABEL"])
fig.tight_layout()
fig.savefig(f"{CHARTS_DIR}/06_delay_by_month.png", dpi=DPI)
plt.show()

print("2015-only EDA charts saved to outputs/charts/ (original filenames, no _v2 suffix)")

## Appendix A.5 — Model (2015 only)

Same features, same leakage exclusions, same imbalance fix as Section 7
above, run against the single-year data.

In [ ]:
model_df_appendix = spark.read.parquet(f"{PROCESSED_DIR}/flights_clean.parquet").select(
    "AIRLINE", "ORIGIN_AIRPORT", "MONTH", "DAY_OF_WEEK",
    "SCHEDULED_DEPARTURE_HOUR", "DISTANCE", LABEL_COL,
)
model_df_appendix.persist(StorageLevel.DISK_ONLY)
print(f"Total rows for modeling (2015 only): {model_df_appendix.count()}")

airline_indexer_appendix = StringIndexer(inputCol="AIRLINE", outputCol="AIRLINE_INDEXED", handleInvalid="keep")
airport_indexer_appendix = StringIndexer(inputCol="ORIGIN_AIRPORT", outputCol="ORIGIN_AIRPORT_INDEXED", handleInvalid="keep")
model_df_appendix = airline_indexer_appendix.fit(model_df_appendix).transform(model_df_appendix)
model_df_appendix = airport_indexer_appendix.fit(model_df_appendix).transform(model_df_appendix)

assembler_appendix = VectorAssembler(inputCols=FEATURE_COLS, outputCol="features")
model_df_appendix = assembler_appendix.transform(model_df_appendix)

train_df_appendix, test_df_appendix = model_df_appendix.randomSplit([0.8, 0.2], seed=42)
train_df_appendix.persist(StorageLevel.DISK_ONLY)
test_df_appendix.persist(StorageLevel.DISK_ONLY)
print(f"Train rows: {train_df_appendix.count()}, Test rows: {test_df_appendix.count()}")

test_on_time_appendix = test_df_appendix.filter(f"{LABEL_COL} == 0").count()
majority_baseline_accuracy_appendix = round(test_on_time_appendix / test_df_appendix.count(), 4)
print(f"\n'Always predict on-time' baseline accuracy on test set: {majority_baseline_accuracy_appendix}")

lr_result_appendix = evaluate(lr.fit(train_df_appendix).transform(test_df_appendix), "LogisticRegression (unweighted, 2015 only)")
rf_model_appendix = rf.fit(train_df_appendix)
rf_result_appendix = evaluate(rf_model_appendix.transform(test_df_appendix), "RandomForest (unweighted, 2015 only)")

print(
    "\n-> Same degeneracy as the combined dataset: both unweighted models are "
    f"functionally identical to the majority-class baseline "
    f"({majority_baseline_accuracy_appendix * 100:.2f}% accuracy)."
)

label_counts_appendix = {int(r[LABEL_COL]): r["count"] for r in train_df_appendix.groupBy(LABEL_COL).count().collect()}
total_train_appendix = sum(label_counts_appendix.values())
class_weights_appendix = {label: total_train_appendix / (2 * count) for label, count in label_counts_appendix.items()}
print(f"\nClass weights (total / (2 * class_count)): {class_weights_appendix}")

train_weighted_appendix = train_df_appendix.withColumn(
    "classWeight", F.when(F.col(LABEL_COL) == 1, class_weights_appendix[1]).otherwise(class_weights_appendix[0])
)
train_weighted_appendix.persist(StorageLevel.DISK_ONLY)
train_weighted_appendix.count()

lr_weighted_predictions_appendix = LogisticRegression(
    featuresCol="features", labelCol=LABEL_COL, weightCol="classWeight"
).fit(train_weighted_appendix).transform(test_df_appendix)
print(f"\nWeighted LogisticRegression AUC (2015 only): {round(BinaryClassificationEvaluator(labelCol=LABEL_COL).evaluate(lr_weighted_predictions_appendix), 4)}")
lr_sweep_appendix, lr_best_appendix = sweep_thresholds(lr_weighted_predictions_appendix, "Weighted LogisticRegression (2015 only)")

rf_weighted_model_appendix = RandomForestClassifier(
    featuresCol="features", labelCol=LABEL_COL, numTrees=50, maxDepth=8,
    maxBins=700, seed=42, weightCol="classWeight",
).fit(train_weighted_appendix)
rf_weighted_predictions_appendix = rf_weighted_model_appendix.transform(test_df_appendix)
rf_weighted_auc_appendix = BinaryClassificationEvaluator(labelCol=LABEL_COL).evaluate(rf_weighted_predictions_appendix)
print(f"\nWeighted RandomForest AUC (2015 only): {round(rf_weighted_auc_appendix, 4)}")
rf_sweep_appendix, rf_best_appendix = sweep_thresholds(rf_weighted_predictions_appendix, "Weighted RandomForest (2015 only)")

print("\n=== Appendix A.5 summary: Before vs After (2015 only) ===")
print(f"BEFORE (unweighted RF, 0.5 threshold): {rf_result_appendix['confusion_matrix']} -> 0 true positives")
print(f"AFTER  (weighted RF, tuned threshold={rf_best_appendix['threshold']}): precision={rf_best_appendix['precision']}, recall={rf_best_appendix['recall']}, f1={rf_best_appendix['f1']}")
print(
    f"\nCompare to the combined 2015+2016 result in Section 7: AUC "
    f"{round(rf_weighted_auc_appendix, 4)} (2015 only) vs 0.649 (combined) — "
    "confirms the scaling-barely-moved-performance finding from both directions."
)